# Machine Learning Classification Pipeline - Kelayakan Tanaman

Notebook ini mencakup langkah end-to-end:
1. Data loading
2. EDA
3. Preprocessing dengan ColumnTransformer
4. Handling imbalance (SMOTE/RandomOverSampler jika tersedia)
5. Train-test split
6. Training Random Forest
7. Hyperparameter tuning
8. Evaluasi
9. Build final pipeline
10. Simpan model ke model/pipeline.pkl

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

try:
    from imblearn.over_sampling import SMOTE, RandomOverSampler
    from imblearn.pipeline import Pipeline as ImbPipeline
    IMBLEARN_AVAILABLE = True
except ImportError:
    IMBLEARN_AVAILABLE = False

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

In [ ]:
ROOT_DIR = Path.cwd().resolve().parent if Path.cwd().name == "notebook" else Path.cwd().resolve()
DATA_PATH = ROOT_DIR / "dataset" / "agro_environmental_dataset.csv"
MODEL_DIR = ROOT_DIR / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Dataset loaded from: {DATA_PATH}")
df.head()

## Step 1 - Data Loading Check

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isna().sum())

## Step 2 - EDA
Analisis cepat distribusi fitur, missing values, duplicate data, target imbalance, dan korelasi numerik.

In [ ]:
TARGET_COLUMN = "failure_flag"
IGNORE_COLUMNS = ["location_id", "suitability_score"]

duplicates_count = df.duplicated().sum()
missing_total = int(df.isna().sum().sum())

print(f"Total missing values: {missing_total}")
print(f"Total duplicate rows: {duplicates_count}")

if TARGET_COLUMN in df.columns:
    target_dist = df[TARGET_COLUMN].value_counts(dropna=False)
    print("\nTarget distribution:")
    print(target_dist)
    imbalance_ratio = target_dist.max() / max(target_dist.min(), 1)
    print(f"Imbalance ratio (major/minor): {imbalance_ratio:.2f}")
else:
    raise ValueError(f"Target column '{TARGET_COLUMN}' tidak ditemukan.")

In [ ]:
numeric_cols_for_plot = df.select_dtypes(include=[np.number]).columns.tolist()
if TARGET_COLUMN in numeric_cols_for_plot:
    numeric_cols_for_plot.remove(TARGET_COLUMN)

sampled_numeric = numeric_cols_for_plot[:8]
if sampled_numeric:
    df[sampled_numeric].hist(figsize=(16, 10), bins=20)
    plt.suptitle("Histogram Fitur Numerik (sample)")
    plt.tight_layout()
    plt.show()

In [ ]:
categorical_cols_for_plot = df.select_dtypes(exclude=[np.number]).columns.tolist()
categorical_cols_for_plot = [c for c in categorical_cols_for_plot if c not in ["location_id"]]

for col in categorical_cols_for_plot[:4]:
    plt.figure(figsize=(10, 4))
    sns.countplot(data=df, x=col, order=df[col].value_counts().index)
    plt.title(f"Countplot - {col}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x=TARGET_COLUMN, order=df[TARGET_COLUMN].value_counts().index)
plt.title("Target Distribution")
plt.tight_layout()
plt.show()

In [ ]:
corr = df.select_dtypes(include=[np.number]).corr(numeric_only=True)
plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Correlation Heatmap (Numeric Features)")
plt.tight_layout()
plt.show()

## Step 3, 4, 5 - Preprocessing, Imbalance Handling, Data Split
Preprocessing dilakukan dengan ColumnTransformer agar konsisten antara training dan inferensi backend.

In [ ]:
df_model = df.drop_duplicates().copy()

feature_columns = [c for c in df_model.columns if c not in IGNORE_COLUMNS + [TARGET_COLUMN]]
X = df_model[feature_columns]
y = df_model[TARGET_COLUMN]

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_transformer = SkPipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_transformer = SkPipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

In [ ]:
base_model = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)

class_counts = y_train.value_counts()
imbalance_ratio = class_counts.max() / max(class_counts.min(), 1)
use_sampler = IMBLEARN_AVAILABLE and imbalance_ratio > 1.5

sampler = None
if use_sampler:
    sampler = SMOTE(random_state=RANDOM_STATE) if class_counts.min() >= 6 else RandomOverSampler(random_state=RANDOM_STATE)

if use_sampler:
    model_pipeline = ImbPipeline(steps=[
        ("preprocessor", preprocessor),
        ("sampler", sampler),
        ("model", base_model),
    ])
else:
    if imbalance_ratio > 1.5:
        base_model.set_params(class_weight="balanced")
    model_pipeline = SkPipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", base_model),
    ])

print(f"Imbalanced handling active: {use_sampler}")
print(f"imblearn available: {IMBLEARN_AVAILABLE}")

## Step 6, 7, 8 - Training, Tuning, Ensemble (Random Forest wajib)

In [ ]:
param_distributions = {
    "model__n_estimators": [100, 200, 300, 400],
    "model__max_depth": [None, 8, 12, 16, 24],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None],
}

search = RandomizedSearchCV(
    estimator=model_pipeline,
    param_distributions=param_distributions,
    n_iter=20,
    cv=5,
    scoring="f1_weighted",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

search.fit(X_train, y_train)
best_pipeline = search.best_estimator_
print("Best params:")
print(search.best_params_)
print(f"Best CV score (f1_weighted): {search.best_score_:.4f}")

## Step 9 - Evaluasi Model

In [ ]:
y_pred = best_pipeline.predict(X_test)

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - Test Set")
plt.tight_layout()
plt.show()

## Step 10, 11 - Final Pipeline dan Save Model

In [ ]:
PIPELINE_PATH = MODEL_DIR / "pipeline.pkl"
joblib.dump(best_pipeline, PIPELINE_PATH)
print(f"Pipeline saved to: {PIPELINE_PATH}")

In [ ]:
sample_payload = X_test.iloc[[0]].to_dict(orient="records")[0]
sample_payload

## Insight Template (Isi setelah run semua cell)
- Distribusi target: tulis proporsi kelas mayoritas/minoritas dari output countplot.
- Missing values: tulis kolom mana yang punya missing dan metode imputasi yang dipakai.
- Duplicate: tulis jumlah duplicate sebelum dan sesudah drop.
- Korelasi: sebutkan fitur yang korelasinya paling menonjol terhadap target/faktor lain.
- Evaluasi model: simpulkan apakah model sudah layak deployment dari metrik Accuracy, Precision, Recall, F1.